In [ ]:
# NEW CONTEXT: HOSPITAL PATIENT DETERIORATION PREDICTION
# Scenario: ICU Patient Health Risk Forecasting
# A hospital ICU continuously monitors critical patient vitals.

# The goal is to predict the patient’s risk score for the next 6 hours 
# using the previous 12 hours of multivariate time-series data, 
# so doctors can intervene before a medical emergency occurs.

In [28]:
import numpy as np
import pandas as pd

In [29]:
import tensorflow as tf

In [30]:
df=pd.read_csv("patient_vitals.csv")
df.head(2)

,Hour,HR,BP,O2,Resp,Temp
0,1,82,120,98,16,36.8
1,2,85,118,97,17,36.9


In [31]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN,Dense,Dropout
from tensorflow.keras.optimizers import Adam
from sklearn.preprocessing import MinMaxScaler

In [32]:
features=["HR","BP","O2","Resp","Temp"]
data=df[features].values

In [33]:
scaler=MinMaxScaler()
data_scaled=scaler.fit_transform(data)

In [34]:
risk=(df["Temp"]*1.0+df["Resp"]*1.2+df["HR"]*0.8+(100-df["O2"])*1.5).values
risk=risk.reshape(-1,1)
risk

array([[124.6],
       [129.8],
       [135. ],
       [141. ],
       [147. ],
       [153.1],
       [159.2],
       [165.3],
       [171.4],
       [180.4],
       [189.3],
       [199.9]])

In [35]:
risk_scaler=MinMaxScaler()
risk_scaled=risk_scaler.fit_transform(risk)

In [49]:
#  Build sequences: 10h input → 2h output
SEQ_IN=4
SEQ_OUT=1

In [50]:
X,y=[],[]

for i in range(len(data_scaled)-SEQ_IN-SEQ_OUT):
    X.append(data_scaled[i:i+SEQ_IN])
    y.append(risk_scaled[i+SEQ_IN:i+SEQ_IN+SEQ_OUT])

X=np.array(X)
y=np.array(y)    

In [51]:
split = int(0.8 * len(X))
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

In [52]:
model=Sequential([
    SimpleRNN(64,return_sequences=True,input_shape=(SEQ_IN,len(features))),
    Dropout(0.2),
    SimpleRNN(32),
    Dropout(0.2),
    Dense(SEQ_OUT)
])

In [53]:
model.compile(optimizer=Adam(0.001), loss="mse")
model.summary()

Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ simple_rnn_8 (SimpleRNN)        │ (None, 4, 64)          │         4,480 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_8 (Dropout)             │ (None, 4, 64)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_9 (SimpleRNN)        │ (None, 32)             │         3,104 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_9 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 7,617 (29.75 KB)

 Trainable params: 7,617 (29.75 KB)

 Non-trainable params: 0 (0.00 B)

In [54]:
history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=50,
    batch_size=16,
    verbose=1
)

Epoch 1/50


1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step - loss: 0.3369 - val_loss: 0.1760
Epoch 2/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 88ms/step - loss: 0.3920 - val_loss: 0.1194
Epoch 3/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 108ms/step - loss: 0.2317 - val_loss: 0.0825
Epoch 4/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 90ms/step - loss: 0.1725 - val_loss: 0.0978
Epoch 5/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 91ms/step - loss: 0.0669 - val_loss: 0.1698
Epoch 6/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 255ms/step - loss: 0.0935 - val_loss: 0.1612
Epoch 7/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 147ms/step - loss: 0.2635 - val_loss: 0.0905
Epoch 8/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 109ms/step - loss: 0.1094 - val_loss: 0.0422
Epoch 9/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 130ms/step - loss: 0.3977 - val_loss: 0.0160
Epoch 10/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 122ms/step - loss: 0.8591 - val_loss: 0.0022
Epoch 11/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 107ms/step - loss: 0.2926 - val_loss: 0.0112
Epoch 12/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 115ms/step - loss: 0.3228 - val_loss: 0.0137
Epoch 

In [55]:
pred_scaled = model.predict(X_test)
pred = risk_scaler.inverse_transform(pred_scaled.reshape(-1, 1))

print("\nPredicted next 6-hour risk scores:")
print(pred[:6])

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 218ms/step

Predicted next 6-hour risk scores:
[[184.36404]
 [191.79034]]
